# Lecture 05 — Frameworks NLP

* hugging face

* langchain

* lightning.ai

* oLLama

* vLLM

* AWS Bedrock

* kaggle

## API KEYs que puede necesitar para ejecutar este notebook

In [ ]:

import os
os.environ["OPENAI_API_KEY"] = "sk-XXX"

# Framework huggingface

## Que es Huggingface?

* Es una plataforma y ecosistema, centrado principalmente COMO REPOSITORIO de modelos NLP, visión, audio y otras señales.
* Herramienta para usar, entrenar y desplegar modelos

    La biblioteca principal es transformers, con muchos modelos como:

    * BERT
    * RoBERTa
    * T5
    * DistilBERT
    * LlaMA
    * Mistral
    * etc

* Datasets, posee un gran banco de datasets para entrenamiento, fine-tuning, pruebas, etc

* Plataforma tipo gihub para que publiques modelos y datasets.

* Plataforma de entrenamiento, inferencia y despliegue

    * transformers → usar y entrenar modelos
    * datasets → cargar grandes datasets
    * Accelerate → entrenar en múltiples GPUs
    * Inference Endpoints → desplegar modelos como APIs sin gestionar servidores

In [ ]:
# ejemplo con hugging face: analisis de sentimientos

from transformers import pipeline
sentiment_analysis = pipeline("sentiment-analysis")
result = sentiment_analysis("I love this product!")
print(result)
result = sentiment_analysis("este producto esta defectuoso!")
print(result)


In [ ]:
# usar modelos especificos:

from transformers import pipeline
pipe = pipeline("sentiment-analysis",    
                model="nlptown/bert-base-multilingual-uncased-sentiment")
resultado = pipe("El servicio fue excelente.")
print(resultado)

In [ ]:
# como listar el conjunto de modelos para una tarea especifica?

# 1. desde la página web de hugging face, buscar por la tarea (ejemplo: sentiment-analysis) y revisar los modelos disponibles
# 2. usar la API de hugging face para listar los modelos disponibles para una tarea específica (ejemplo: usando la biblioteca transformers para listar los modelos de análisis de sentimientos)
from transformers import pipeline
from transformers import AutoModelForSequenceClassification, AutoTokenizer      
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
sentiment_analysis = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
result = sentiment_analysis("I love this product!")
print(result)
result = sentiment_analysis("este producto esta defectuoso!")
print(result)       


In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits

predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]


In [ ]:
from transformers import pipeline

# Load the classification pipeline with the specified model
pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

# Classify a new sentence
sentence = "I love this product! It's amazing and works perfectly."
result = pipe(sentence)

# Print the result
print(result)


In [ ]:
# listar modelos desde código:

from huggingface_hub import list_models

# opcion 1:
# modelos = list(list_models(filter="sentiment-analysis"))

# opción 2:

modelos = list(list_models(filter="sentiment-analysis",    
                      sort="downloads",   # también puedes usar "likes"    
                      direction=-1        # -1 = descendente (los más populares primero)
                      ))

print(f"Total modelos encontrados: {len(modelos)}")

for m in modelos[:10]:
    print(m.modelId)


mini proyecto:

usando los datos de ag_news, realizar el mismo clasificador del taller1, pero con huggingface y modelos preentrenados de transformers.

realizar una comparación de rendimiento entre los modelos clasicos NB y RL con Hugging Face.

Realizar una comparación de rendimeinto entre varios o como seleccionaría los modelos mejores de transformers para estos datos y labels.



In [ ]:
''' recordar el proceso clasico:

    1. textos originales (datasets ag_news etiquetados)
    2. preprocesamiento (tokenización, limpieza, etc.)
    3. Vectorización (BoW, TF-IDF, word embeddings, etc.) porque aun aca hay WE?
    4. entrenar un modelo con NB y RL
    5. evaluar con precision, recall, f1, etc.

    visión moderna:

    1. textos originales (datasets ag_news etiquetados)
    2. Tokenizador con un modelo preentrenado (ejemplo: BERT tokenizer, DistilBERT tokenizer, etc.) (se hace o no preprocesamiento?)
    3. cargar un modelo preentrenado (ejemplo: BERT, DistilBERT, etc.) y usarlo para generar predicciones directamente (sin necesidad de entrenar un modelo desde cero)
    
'''

''' flujo de trabajo:

    1. cargar el dataset AG NEWS usando 'datasets' de hugging face
    2. preprocesar el dataset usando un tokenizador de un modelo preentrenado
    3. elegir un modelo preentrenado (ejemplo: DistilBERT) y usarlo para generar predicciones directamente
    4. tokenizar
    5. crear el modelo de clasificación (AutoModelForSequenceClassification)
    6. entrenar con Trainer de transformers
    7. evaluar con precision, recall, f1, etc. y comparar los resultados con un modelo clásico (ejemplo: Naive Bayes y Reg Logistica)

'''

In [ ]:
# instalar dependencias:
!pip3 install transformers datasets huggingface_hub accelerate

In [ ]:
# 2 cargar el dataset AG NEWS usando 'datasets' de hugging face

from datasets import load_dataset

dataset = load_dataset("ag_news")

print(dataset)
# normalmente tiene 'train' y 'test'
print(dataset["train"][0])

In [ ]:
# seleccionar un modelo y su tokenizador:

from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"  # podrías cambiarlo luego
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
# tokenizar el dataset:

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],          # o el campo que corresponda ('text', 'description'...)
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_batch, batched=True)

In [ ]:
print(tokenized_dataset)

print(tokenized_dataset["train"])

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])  # dejamos solo inputs + label
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

In [ ]:
# crear el modelo de clasificación:

from transformers import AutoModelForSequenceClassification

num_labels = 4  # AG News tiene 4 clases

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels
)

In [ ]:
!pip3 list | grep transformers

In [ ]:
import transformers
print("Ruta del paquete:", transformers.__file__)
print("Versión:", transformers.__version__)
from transformers import TrainingArguments
print(TrainingArguments)

In [ ]:
import transformers
print("Ruta del paquete:", transformers.__file__)
print("Versión:", transformers.__version__)

from transformers import TrainingArguments
from inspect import signature
print("Firma de __init__:", signature(TrainingArguments.__init__))

In [ ]:
# configurar entrenamiento con Trainer

from transformers import TrainingArguments, Trainer
import numpy as np

#from datasets import load_metric
#metric = load_metric("accuracy")

import evaluate
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):    
    logits, labels = eval_pred    
    preds = np.argmax(logits, axis=-1)    
    return metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./agnews-transformer",
    eval_strategy="epoch",   
# en versiones anteriores: evaluation_strategy="epoch",
    save_strategy="epoch",    
    learning_rate=2e-5,    
    per_device_train_batch_size=16,    
    per_device_eval_batch_size=16,    
    num_train_epochs=2,    
    weight_decay=0.01,
    )


In [ ]:
# 7. Crear Trainer (análogo a tu “entrenador” de Regresión Logística)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

trainer.train() 


In [ ]:
results = trainer.evaluate()
print(results)

# Langchain framework

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

model = ChatOpenAI()
prompt = ChatPromptTemplate.from_template("Explain {topic} in simple terms")
chain = prompt | model

chain.invoke({"topic": "Frameworks NLP to build real applications"})

In [ ]:
# cargar modelos

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

In [ ]:
#agentes con langchain

from langchain.agents import initialize_agent, Tool

### Aplicación chat con LangChain



In [ ]:
# aplicación chatbot basico con LangChain:

!pip install -U langchain langchain-openai faiss-cpu tiktoken python-dotenv

OPENAI_API_KEY="TU_API_KEY"

import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def main():
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY en tu .env")

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

    prompt = ChatPromptTemplate.from_messages([
        ("system", "Eres un asistente útil y conciso. Si no sabes algo, dilo."),
        ("human", "{question}")
    ])

    chain = prompt | llm | StrOutputParser()

    print("Chat básico LangChain (escribe 'exit' para salir)\n")
    while True:
        q = input(">>> ").strip()
        if q.lower() in {"exit", "quit"}:
            break
        ans = chain.invoke({"question": q})
        print(f"ChatBot: {ans}\n")

if __name__ == "__main__":
    main()

In [ ]:
!python app_chat.py

### Aplicación chat con RAG y FAISS local como vector storage

    docs/*.txt
    app_rag.py
    .env


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def load_docs(folder: str):
    texts = []
    for p in Path(folder).glob("*.txt"):
        texts.append(p.read_text(encoding="utf-8"))
    if not texts:
        raise RuntimeError(f"No encontré .txt en {folder}")
    return "\n\n".join(texts)

def build_vectorstore(raw_text: str):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120
    )
    chunks = splitter.split_text(raw_text)

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vs = FAISS.from_texts(chunks, embedding=embeddings)
    return vs

def main():
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY en tu .env")

    # 1) Ingesta
    corpus = load_docs("docs")
    vectorstore = build_vectorstore(corpus)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    # 2) LLM
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

    # 3) Prompt RAG
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Responde usando SOLO el contexto. "
         "Si el contexto no alcanza, di: 'No está en los documentos'."),
        ("human",
         "Pregunta: {question}\n\n"
         "Contexto:\n{context}")
    ])

    # 4) Loop interactivo
    print("RAG básico LangChain + FAISS (escribe 'exit' para salir)\n")
    while True:
        q = input("Tú: ").strip()
        if q.lower() in {"exit", "quit"}:
            break

        docs = retriever.get_relevant_documents(q)
        context = "\n\n---\n\n".join([d.page_content for d in docs])

        chain = prompt | llm | StrOutputParser()
        ans = chain.invoke({"question": q, "context": context})

        print(f"Bot: {ans}\n")

if __name__ == "__main__":
    main()

In [ ]:
!python app_rag.py